In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v1_0 import BioJepa, BioJepaConfig
from training_v1_0 import create_model, maybe_compile
from config_v1_0 import DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = False
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/jepa/v1_0').expanduser()
ref_root = Path('~/data/jepa/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir = ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


## Stage 0: Initialize Model 

In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=5.699,
    film_linear_multiple=0.6769,
    sim_coeff=50.18,
    std_coeff=25.44,            # sim_coeff * std_to_sim_ratio (0.5069)
    cov_coeff=0.5158,           # sim_coeff * cov_to_sim_ratio (0.01028)
    pert_latent_dim=128,
    pert_mode_dim=64,
    predictor_embed_dim=128,
    predictor_n_layer=4,
    predictor_heads=4,
)


EVAL_BATCH_SIZE = 64

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

Student/Teacher: 7,979,650
ACpredictor: 2,565,760
PerturbationComposer: 444,224


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_ac_v1_0.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

## Stage 1: Eval
### Alignment Training Eval

In [6]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
align_eval_results = run_composer_evals(align_eval_ctx)

Using cuda
Loaded 10791 v1.0 alignment pairs
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Encoded DNA sequences: torch.Size([11643, 128])
Encoded chemical sequences: torch.Size([188, 128])
Loaded target bank: torch.Size([9975, 320])
Encoded protein targets: torch.Size([9975, 128])
seq_to_target_retrieval: dna_mrr=0.0034
cross_modality_target_consistency: Within=0.6420, Between=0.5702, Ratio=1.13x
seq_target_gap_analysis: dna_gap=1.13
paired_alignment_quality: dna_sim=0.3806
mode_sensitivity: Classification_acc=0.8286 (5.8x chance)
mode_semantic_consistency: semantic_gap=0.1317, cross_mode_mrr=0.0678
fusion_quality: Fused_var=0.2533, Seq_var=0.1729, Target_var=0.0647
missing_data_robustness: Fused_MRR=0.2145, Seq_only=0.0074, Target_only=1.0000
found 135 shards for split test


multi_pert_alignment: Scanning for multi-pert: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:04<00:00, 115.65it/s]


Loaded 27264 HGNC gene family annotations
target_family_probing: seq_only=0.0885, target_only=0.2289, fused=0.2199
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
action_vector_pathways DNA: ratio=1.0015739496011622


In [7]:
save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

Saved report to /home/ubuntu/data/jepa/v1_0/eval_results/composer_eval_report.json


{'seq_to_target_retrieval': {'config': {'n_targets': 9975},
  'by_modality': {'dna': {'mrr': 0.003408447659481588,
    'median_rank': 4285.5,
    'mean_rank': 4451.587864534336,
    'n_queries': 10630,
    'n_targets': 9975,
    'recall_at_k': {'1': 0.000658513640639699,
     '5': 0.002916274694261524,
     '10': 0.005738476011288805,
     '20': 0.010630291627469425,
     '50': 0.02323612417685795}}}},
 'cross_modality_target_consistency': {'config': {'n_valid_targets': 921,
   'n_within_pairs': 1408,
   'n_between_pairs': 5000},
  'metrics': {'within_target_sim': 0.6420363187789917,
   'between_target_sim': 0.5702140501886607,
   'consistency_ratio': 1.1259566798933978}},
 'seq_target_gap_analysis': {'target_variance': 8.417330741882324,
  'n_targets': 9975,
  'dna': {'seq_variance': 23.172298431396484,
   'centroid_distance': 5.208191871643066,
   'mean_within_seq': 6.754885765388936,
   'mean_seq_to_target': 7.618014812469482,
   'gap_ratio': 1.1277784816884833,
   'n_sequences': 11

In [8]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()


### Encoder Training Evals

In [9]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
encoder_eval_results = run_encoder_evals(eval_ctx)

Using cuda
found 135 shards for split test


batch_invariance: Extracting embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5400/5400 [11:14<00:00,  8.01it/s]


Training classifiers...
batch_invariance: Batch=0.1127 (48.7x), Pert=0.0307 (34.0x)
batch_invariance summary: global_ratio=0.272, within_dataset_macro_ratio=0.292
gene_embedding_pathways: KEGG ratio=1.0435
essential_gene_prediction: Pearson=0.2631, AUROC=0.6669
found 135 shards for split test


cell_type_probing: Extracting embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5400/5400 [11:13<00:00,  8.01it/s]


Training cell type classifier...
cell_type_probing: Accuracy=0.8868 (3.5x chance), Macro F1=0.6325
found 135 shards for split test


...

Training perturbation detector...
perturbation_detection: AUROC=0.5401, Accuracy=0.5308
found 135 shards for split test


embedding_consistency: Extracting embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5400/5400 [11:15<00:00,  8.00it/s]


embedding_consistency: Computing intra-distances for 1429 perturbations...
embedding_consistency: Computing 5000 inter-distances...
embedding_consistency: Intra=16.7546, Inter=14.5280, Ratio=0.87x
embedding_consistency: Computing for dataset adamson...
embedding_consistency: Computing for dataset k562e_raw...
embedding_consistency: Computing for dataset k562gw...
embedding_consistency: Computing for dataset norman...
embedding_consistency: Computing for dataset rep1e...
embedding_consistency: Computing for dataset sciplex...
found 135 shards for split test


latent_space_health: Extracting embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5400/5400 [11:11<00:00,  8.04it/s]


latent_space_health: Eff_dim_90=33/256, Mean_var=0.5293, Isotropy=0.000027
permutation_invariance [adamson]: mean=1.000000, min=1.000000
permutation_invariance [k562e_raw]: mean=1.000000, min=1.000000
permutation_invariance [k562gw]: mean=1.000000, min=1.000000
permutation_invariance [norman]: mean=1.000000, min=0.999999
permutation_invariance [rep1e]: mean=1.000000, min=1.000000
permutation_invariance [sciplex]: mean=1.000000, min=1.000000
permutation_invariance overall: mean=1.000000, min=0.999999


In [10]:
save_report(encoder_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
encoder_eval_results

Saved report to /home/ubuntu/data/jepa/v1_0/eval_results/encoder_eval_report.json


{'batch_invariance': {'config': {'samples': 345600,
   'embedding_dim': 256,
   'num_batches': 432,
   'num_perturbations': 1106},
  'batch_classifier': {'accuracy': 0.1126880787037037,
   'chance': 0.0023148148148148147,
   'above_chance_ratio': 48.681250000000006},
  'perturbation_classifier': {'accuracy': 0.03070023148148148,
   'chance': 0.0009041591320072332,
   'above_chance_ratio': 33.95445601851852},
  'invariance_ratio': 0.2724354859417127,
  'by_dataset': {'k562e_raw': {'config': {'samples': 46506,
     'embedding_dim': 256,
     'num_batches': 48,
     'num_perturbations': 286},
    'batch_classifier': {'accuracy': 0.20973984089443132,
     'chance': 0.020833333333333332,
     'above_chance_ratio': 10.067512362932703},
    'perturbation_classifier': {'accuracy': 0.012900451515803053,
     'chance': 0.0034965034965034965,
     'above_chance_ratio': 3.6895291335196734},
    'invariance_ratio': 0.061506919528446946},
   'k562gw': {'config': {'samples': 189095,
     'embedding_d

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Full Model Eval

In [13]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_ac_v1_0_decoder.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

<All keys matched successfully>

In [14]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED,
})
full_eval_results = run_ac_evals(eval_ctx)

Using cuda
found 135 shards for split test


Running test inference:   0%|                                                                                                                                                        | 0/5400 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


Running test inference:   1%|█                                                                                                                                            | 40/5400 [00:20<4:26:21,  2.98s/it]/home/ubuntu/code/biojepa/evals/evals.py:654: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_top, t_top)
Running test inference: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5400/5400 [44:30<00:00,  2.02it/s]


Aggregated 1416 single-pert, 13 multi-pert perturbations, 345600 samples, 135 shards
  adamson: 11 perturbations, 4288 samples
  k562e_raw: 286 perturbations, 46506 samples
  k562gw: 1053 perturbations, 189095 samples
  norman: 10 perturbations, 9472 samples
  rep1e: 287 perturbations, 22838 samples
  sciplex: 54 perturbations, 73401 samples
Cached test inference to /home/ubuntu/data/jepa/v1_0/test_inference_cache (135 shards)
expression_prediction: Pearson=0.9873, R2=0.9680, Centroid_acc=0.1492
gene_level_analysis: Dir_acc=0.9821, Top50_acc=0.7185


perturbation_retrieval (dna): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [2:17:51<00:00, 41.36s/it]


perturbation_retrieval (dna): MRR=0.0007


perturbation_retrieval (chemical): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 54/54 [00:35<00:00,  1.50it/s]


perturbation_retrieval (chemical): MRR=0.0579
Loaded dataset gene masks: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
uncertainty_calibration: ECE=0.3232, Monotonicity=66.67%
moa_matching expression: Within=0.3240, Between=0.3084, Gap=0.0156, Ratio=1.0505x
moa_matching latent: Within=0.6982, Between=0.6930, Gap=0.0052, Ratio=1.0074x
Loaded Norman combo mapping: 132 combos
Loaded Norman single-gene deltas: 105 genes
Loaded Norman GI subtypes: 88 combos
Loaded dataset splits: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
combination_perturbation: 13 combo perts, 4283 samples, 13 additive baseline, 5 GI-labeled, 13 generalization-classified
dose_response: monotonicity=50.00%, real_mono=48.77%, spearman=-0.0212, curve_sim=0.5237164240225922


In [15]:
save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

Saved report to /home/ubuntu/data/jepa/v1_0/eval_results/ac_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1416,
   'genes': 10000,
   'test_samples': 345600},
  'sample_level': {'mse': 0.19492396023248418,
   'pearson_r_top20': 0.8654291432278572},
  'perturbation_level': {'r2_all_genes': {'mean': 0.9679888112649406,
    'median': 0.9806722104549408},
   'r2_top50_degs': {'mean': 0.8077261285003969, 'median': 0.8957716226577759},
   'mse': {'mean': 0.007274444680660963, 'median': 0.004187296144664288},
   'pearson_all_genes': {'mean': 0.987335892089006,
    'median': 0.9932138025760651},
   'pearson_delta_all_genes': {'mean': 0.33241361749390047,
    'median': 0.3330247551202774},
   'pearson_top50_degs': {'mean': 0.599334994933468,
    'median': 0.6364593803882599}},
  'centroid_accuracy': {'accuracy': 0.1492087415222306, 'n_groups': 1327},
  'vs_baseline': {'beat_rate': 0.2860169491525424, 'n_evaluated': 1416},
  'severity': {'pearson_r': 0.7974816560745239,
   'spearman_r': 0.7807355552907821},
  'error_by_magnitude': {'0-0.25'

In [16]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()